In [1]:
using DelimitedFiles
using CairoMakie
using GLMakie
using LinearAlgebra
using Random
using CSV
using DataFrames
using Serialization

In [2]:
Randomisation = false

save_h_matrix = false
read_h_matrix = false
z_boundary_conditions = false  # if true z_boundary_conditions = true: the periodic boundary conditions are switched on. 

visulize_sim_box = true

copy_size::Int64 = 27 # 27
nx::Int64 = 10    #40#copy_size #24
ny::Int64 = nx    #40#copy_size #24
nz::Int64 = 6     #10#copy_size #15

Interaction_radius_cutoff = true
interaction_cut_off_radius = 4   # interaction radius cut-off in units of number of unit cells 
print("cut-off radius = ", interaction_cut_off_radius * a0_CO, "\n")

include("./introduction.jl")
include("./ir_spectra.jl")
include("./ir_spectra_h_matrix.jl")
include("./ir_spectra_centre.jl")

########################## Read the measured FTIR data ##########################
file_path_p = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_p_pol.txt")
file_p      = open(file_path_p)
header_p    = split(strip(readline(file_p)), '\t')
data_p      = readdlm(file_p, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_p = data_p[:, 1]
A_p = data_p[:, 2]
 
file_path_s = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_s_pol.txt")
file_s      = open(file_path_s)
header_s    = split(strip(readline(file_s)), '\t')
data_s      = readdlm(file_s, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_s = data_s[:, 1]
A_s = data_s[:, 2];

UndefVarError: UndefVarError: `a0_CO` not defined

In [16]:
ν0 = 2136.97 # cm-1 #2050.0
νk::Vector{Float64} = collect(ν0- 1*range :step:ν0 + 1*range)
nmols_ml = 4*nx*ny*nz

if Randomisation == true    
    Random.seed!(1234)
    num = sign.(rand(nmols_ml) .- 0.5)
    eu_unit_vector = num .* eu # randomised unit vector
else
    eu_unit_vector = eu
end

2400-element Vector{Vector{Float64}}:
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 [-1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 [-1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 ⋮
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 [-1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 [-1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0]

In [17]:
# @time ipda, isda, ip, is = ir_spectra(νk, eu_unit_vector, com_ol, Δν)

# fig = Figure(size=(900, 600))

# ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
# ax.xlabelsize, ax.ylabelsize  = 24, 24
# ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
# ax.xticklabelsize, ax.yticklabelsize = 20, 20

# conversion_to_mOD = 1000

# lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
# lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

# Normalisation_exciton = maximum(ipda*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
# lines!(ax, νk, ipda/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
# lines!(ax, νk, isda/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")
# #lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
# #lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

# axislegend(ax, labelsize = 18, position=:lt)
# DataInspector(fig)
# display(fig)

In [18]:
@time ipda_h_matrix, isda_h_matrix, ip_h_matrix, is_h_matrix, h_matrix = ir_spectra_h_matrix(νk, eu_unit_vector, com_ol, Δν)

Matrix{Float64}
(2400, 2400)
2
2400
  3.915913 seconds (182.13 M allocations: 6.272 GiB, 8.93% gc time, 3.30% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2138.641578984952 -0.8750036072813138 … 0.0015831910478301436 -0.006416090035943211; -0.8750036072813138 2138.641165937677 … -0.005881929163354031 0.004119731762187237; … ; 0.0015831910478301436 -0.005881929163354031 … 2138.579508353969 -0.8750036072813138; -0.006416090035943211 0.004119731762187237 … -0.8750036072813138 2138.590562653795])

In [19]:
h_matrix_path = joinpath("h-matrix", "h-matrix_" *string(nx)*"x"*string(ny)*"x"*string(nz)*"_"*"z_bound_"*string(z_boundary_conditions)*".bin")
# save h matrix as binary file
if save_h_matrix == true
    # save txt file in folder of force matrix # writedlm(h_matrix_path, h_matrix)
    # Save to a binary file
    open(h_matrix_path, "w") do io
        serialize(io, h_matrix)
        println("Matrix saved in binary format as 'matrix.bin'")
    end
end

In [20]:
# Load from binary file
if read_h_matrix == true
    h_matrix_large = open(h_matrix_path, "r") do io # takes about 1 min after restart
        deserialize(io)
    end
    # println("Matrix loaded: ", h_matrix_large)    
    # get the diagonal elements for the different energies
    h_matrix_large_energies = diag(h_matrix_large)
    fig = Figure(size=(900, 600))
    ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
    lines!(ax, h_matrix_large_energies, color=:red, label = L"energies")
    axislegend(ax, labelsize = 18, position=:lt)
    DataInspector(fig)
    display(fig)
end

In [21]:
small_box_size_fraction = 5
# get the index of each molecules and define the middle position of cuboid 
nx_centre::Int64 = nx÷(small_box_size_fraction+0.00000)   # get subset of alpha-CO which is smaller than small_box_size_fraction-times the size of the total simulation box (use modulo ÷ to get unit cell below exactly small_box_size_fraction the size)
ny_centre::Int64 = ny÷(small_box_size_fraction+0.00000) 
nz_centre::Int64 = nz

println(nx_centre,' ', ny_centre,' ', nz_centre)
nmols_centre = 4*nx_centre*ny_centre*nz_centre

2 2 6


96

In [22]:
max_x_centre = maximum(com_ol[:, 1]) * nx_centre / nx
max_y_centre = maximum(com_ol[:, 2]) * ny_centre / ny
max_z_centre = maximum(com_ol[:, 3]) * nz_centre / nz
print(max_x_centre, ' ', max_y_centre, ' ', max_z_centre)

1.9 1.9 5.5

In [23]:
# indexing of molecules in the centre of the simulation box:
com_ol_centre         = zeros(Float64, nmols_centre, 3) 
eu_unit_vector_centre = Vector{Vector{Float64}}(undef, nmols_centre)
large_h_matrix_sum    = zeros(Float64, nmols_centre)

count = 1
for row_i in 1:size(com_ol, 1)
    if com_ol[row_i, 1] <= max_x_centre && com_ol[row_i, 2] <= max_y_centre && com_ol[row_i, 3] <= max_z_centre
        com_ol_centre[count, 1] = com_ol[row_i, 1]
        com_ol_centre[count, 2] = com_ol[row_i, 2]
        com_ol_centre[count, 3] = com_ol[row_i, 3]
        eu_unit_vector_centre[count] = eu_unit_vector[row_i]
        large_h_matrix_sum[count] = h_matrix[row_i, row_i]
        count += 1
    end
end

In [28]:
if visulize_sim_box == true  # visualize the simulation box:

    X = com_ol[:, 1]
    Y = com_ol[:, 2]
    Z = com_ol[:, 3]

    U = [vec[1] for vec in eu_unit_vector]
    V = [vec[2] for vec in eu_unit_vector]
    W = [vec[3] for vec in eu_unit_vector]

    index_vector_of_centre_molecules = [1:nmols_centre;]

    X_centre = com_ol_centre[:, 1]
    Y_centre = com_ol_centre[:, 2]
    Z_centre = com_ol_centre[:, 3]

    U_centre = [vec[1] for vec in eu_unit_vector_centre]
    V_centre = [vec[2] for vec in eu_unit_vector_centre]
    W_centre = [vec[3] for vec in eu_unit_vector_centre]

    # Create a figure and 3D axis
    fig = Figure()
    ax = Axis3(fig[1, 1], xlabel = "X Coordinate", ylabel = "Y Coordinate", zlabel = "Z Coordinate")

    arrows!(ax, vec(X), vec(Y), vec(Z), 0.2 .* vec(U), 0.2 .* vec(V), 0.2 .* vec(W), arrowsize=0.1, color=:red)
    scatter!(ax, vec(X_centre), vec(Y_centre), vec(Z_centre), markersize=40)
    arrows!(ax, vec(X_centre), vec(Y_centre), vec(Z_centre), 0.2 .* vec(U_centre), 0.2 .* vec(V_centre), 0.2 .* vec(W_centre), arrowsize=0.15, color=:blue)

    ax.title = "3D Grid of Molecules with Orientation Arrows"
    # Display the figure
    display(fig)
end

GLMakie.Screen(...)

In [25]:
Interaction_radius_cutoff = true
interaction_cut_off_radius = 4   # m interaction radius cut-off in # of unit cells

# read the force matrix for the maximum size and do eigen(h) operation of small simulation box with entries of the large h matrix on the diagonal
@time ipda, isda, ip, is, eigenvecs = ir_spectra_centre(νk, eu_unit_vector_centre,  com_ol_centre, Δν, nmols_centre, large_h_matrix_sum)

96Matrix{Float64}
(96, 96)
2
96
  0.218506 seconds (2.73 M allocations: 52.010 MiB, 69.98% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  1.4402947237417948e-40, 5.4918402888515534e-42, 1.9810800823714588e-43, 6.760888407128644e-45, 2.1828460461984166e-46, 6.667454299244796e-48, 1.926702188650669e-49, 5.267284005248487e-51, 1.3623116857875775e-52, 3.3333723577233333e-54], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  1.2249955835244056e-40, 4.6709054122064145e-42, 1.6849430356282309e-43, 5.750253266717194e-45, 1.8565486073387563e-46, 5.670786220810502e-48, 1.6386938618987886e-49, 4.479917113680095e-51, 1.1586699163516195e-52, 2.835091524757106e-54], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  4.285283614086247e-40, 1.633977652455208e-41, 5.894273129075508e-43, 2.0115554169131595e-44, 6.494584145918769e-46, 1.9837561929764073e-47, 5.7324839051282814e-49, 1.5671659726350185e-50, 4.0532626194209814e-52, 9.917725739322367e-54], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  4.382509102482605e-44, 1.6710495625983166e-45, 6.028002769896905e-47, 2.0

In [26]:
α = 0*degrees 

ipda_α = (cos(α))^2 .* ipda + (sin(α))^2 .* isda 
isda_α = (cos(α))^2 .* isda + (sin(α))^2 .* ipda;

In [27]:
# CairoMakie.activate!()
# GLMakie.activate!()
fig = Figure(size=(900, 600))

ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
ax.xlabelsize, ax.ylabelsize  = 24, 24
ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
ax.xticklabelsize, ax.yticklabelsize = 20, 20

conversion_to_mOD = 1000

lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

Normalisation_exciton = maximum(ipda*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
lines!(ax, νk, ipda/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
lines!(ax, νk, isda/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")

#lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
#lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

axislegend(ax, labelsize = 18, position=:lt)
DataInspector(fig)
display(fig)

GLMakie.Screen(...)